# 04. VAR, Geweke Causality and Transmission

The previous notebook showed whether the dependence structure changed over time. This notebook studies a more directional question: **how shocks are transmitted across markets**. If the structure of risk changed after COVID-19, then the transmission of shocks between equities, rates, commodities, FX and credit should also look different.

We use a **VAR(2)** exactly as requested, with the following five variables:
- S&P 500,
- US 10Y yield change,
- Oil futures,
- EURUSD,
- US HY Bonds.

We then compute:
- Granger causality as a first reduced-form lead-lag check,
- Geweke causality measures in the time domain,
- Cholesky impulse response functions.

## 1. Setup

We again rely on the same aligned daily return dataset. This keeps the whole project internally consistent: the VAR and Geweke measures should be interpreted as a continuation of the stylized facts, GARCH and DCC evidence rather than as a separate dataset.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from project2_var_utils import (
    VAR_COLUMNS,
    VAR_LABELS,
    VAR_LAG,
    load_var_samples,
    fit_var,
    var_summary_row,
    granger_table,
    geweke_causality_table,
    plot_irf_panel,
    save_var_outputs,
)
from project2_data_utils import ensure_output_dirs, save_figure

sns.set_theme(style="whitegrid", context="talk")
pd.options.display.float_format = "{:.4f}".format
ensure_output_dirs()

## 2. Load the five-variable VAR dataset

The five-variable system is chosen to represent the main channels of market risk discussed in the course: equity, rates, commodities, FX and credit. Because the project studies whether the structure of risk changed after COVID, we estimate the VAR separately before and after the break.

In [ ]:
aligned_var_data, pre_covid, post_covid = load_var_samples()

print(f"Full VAR sample: {aligned_var_data.shape}")
print(f"Pre-COVID VAR sample: {pre_covid.shape}")
print(f"Post-COVID VAR sample: {post_covid.shape}")

aligned_var_data.head()

## 3. Estimate the VAR(2) models

We keep the lag order fixed at 2, exactly as stated in the project instructions. This is useful because it avoids turning the notebook into a lag-selection exercise and makes the pre/post comparison cleaner. The object of interest is not the perfect lag order, but whether the transmission structure changes across the two regimes.

In [ ]:
var_pre = fit_var(pre_covid, lag_order=VAR_LAG)
var_post = fit_var(post_covid, lag_order=VAR_LAG)

var_summary = pd.DataFrame([
    var_summary_row(var_pre, "Pre-COVID"),
    var_summary_row(var_post, "Post-COVID"),
])
var_summary

## 4. Granger causality as a first transmission check

Granger causality is a natural starting point because it answers a simple question: do past values of one market help forecast another, conditional on the rest of the system? This is not structural causality, but it is a useful reduced-form indicator of transmission.

In [ ]:
granger_results = pd.concat([
    granger_table(var_pre, "Pre-COVID"),
    granger_table(var_post, "Post-COVID"),
], axis=0, ignore_index=True)

granger_results.head(20)

## 5. Geweke causality measures

Geweke causality provides a richer decomposition than a simple Granger test. It separates:
- **directional causality**, which measures how much one variable improves the forecast of another;
- **instantaneous causality**, which measures contemporaneous dependence left in the VAR residual covariance matrix.

This matters for the research question because a post-COVID change in the structure of risk may show up not only in lagged spillovers, but also in stronger contemporaneous transmission across markets.

In [ ]:
geweke_results = pd.concat([
    geweke_causality_table(var_pre, pre_covid, "Pre-COVID", lag_order=VAR_LAG),
    geweke_causality_table(var_post, post_covid, "Post-COVID", lag_order=VAR_LAG),
], axis=0, ignore_index=True)

geweke_results.head(20)

In [ ]:
directional_geweke = geweke_results.loc[geweke_results["measure"] == "directional"].copy()
instantaneous_geweke = geweke_results.loc[geweke_results["measure"] == "instantaneous"].copy()

print("Directional Geweke measures")
directional_geweke.head(20)

In [ ]:
print("Instantaneous Geweke measures")
instantaneous_geweke.head(20)

## 6. Cholesky impulse response functions

The Geweke measures summarize how much forecast content and contemporaneous dependence are present in the system. The impulse response functions answer the complementary question: **what is the dynamic path of a shock once it hits the system?**

We use Cholesky orthogonalization in the fixed variable order of the VAR. This remains a reduced-form identification device, but it is standard in this type of empirical exercise and sufficient for comparing pre- and post-COVID transmission patterns.

In [ ]:
irf_pre_figure = plot_irf_panel(var_pre, "Pre-COVID")
save_figure(irf_pre_figure, "04_irf_pre_covid.png")
plt.show()

In [ ]:
irf_post_figure = plot_irf_panel(var_post, "Post-COVID")
save_figure(irf_post_figure, "04_irf_post_covid.png")
plt.show()

## 7. Save the transmission outputs

We save the VAR summary, the Granger results and the Geweke tables so that the regime notebook can reuse the main transmission evidence without re-estimating everything manually.

In [ ]:
save_var_outputs(var_summary, granger_results, geweke_results)
print("Saved VAR and Geweke outputs under outputs/project2/")

## 8. Main takeaways from notebook 04

This notebook studies whether transmission channels changed after COVID-19. The Granger results indicate whether lagged spillovers became stronger or changed direction, the Geweke measures summarize directional and instantaneous dependence, and the IRFs show whether shocks propagate differently through the system after the COVID break.